In [5]:
# %% [markdown]
# # Avaliação de Qualidade de Respostas Geradas - Chatbot Código da Estrada
# Este notebook compara as respostas geradas pelo modelo com as respostas corretas (gold standard) usando métricas BLEU, ROUGE e BERTScore.

# %%
# Instalação (se necessário)
!pip install -q nltk rouge-score bert-score sentence-transformers

# %%
import json
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

nltk.download('punkt')

# %%
# Carrega o JSON com as perguntas, respostas gold e geradas
with open("avaliacao_dataset.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

# %%
# Inicializar o ROUGE scorer
rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# Inicializar modelo para BERTScore com embeddings (alternativa ao modelo com erro)
bert_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# %%
bleu_scores = []
rouge1_scores = []
rougeL_scores = []
bert_f1s = []

# %%
# Cálculo de métricas
for entrada in dados:
    referencia = entrada["gold_variants"]
    gerado = entrada["gerado"]

    # BLEU
    bleu = sentence_bleu([ref.split() for ref in referencia], gerado.split(),
                         smoothing_function=SmoothingFunction().method1)
    bleu_scores.append(bleu)

    # ROUGE (média entre as referências)
    rouge_1_temp = []
    rouge_L_temp = []
    for ref in referencia:
        s = rouge.score(ref, gerado)
        rouge_1_temp.append(s['rouge1'].fmeasure)
        rouge_L_temp.append(s['rougeL'].fmeasure)
    rouge1_scores.append(sum(rouge_1_temp) / len(rouge_1_temp))
    rougeL_scores.append(sum(rouge_L_temp) / len(rouge_L_temp))

    # Similaridade semântica (substituto do BERTScore com fallback robusto)
    emb_gerado = bert_model.encode(gerado, convert_to_tensor=True)
    melhores_f1 = []
    for ref in referencia:
        emb_ref = bert_model.encode(ref, convert_to_tensor=True)
        score = util.cos_sim(emb_gerado, emb_ref).item()
        melhores_f1.append(score)
    bert_f1s.append(max(melhores_f1))

# %%
# Mostrar médias finais
print("\n===== MÉTRICAS MÉDIAS =====")
print(f"BLEU: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"ROUGE-1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"ROUGE-L: {sum(rougeL_scores)/len(rougeL_scores):.4f}")
print(f"BERTScore aprox. (cosine sim): {sum(bert_f1s)/len(bert_f1s):.4f}")

# %%
# Opcional: visualizar os piores exemplos
print("\nExemplos com menor BERTScore:")
ordenado = sorted(zip(bert_f1s, dados), key=lambda x: x[0])
for i, (score, exemplo) in enumerate(ordenado[:3]):
    print(f"\n#{i+1} - BERTScore (aprox) F1: {score:.4f}")
    print("Pergunta:", exemplo["input"])
    print("Esperado:", exemplo["gold_variants"][0])
    print("Gerado:", exemplo["gerado"])


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CarlosMoutinho(11408\AppData\Roaming\nltk_dat
[nltk_data]     a...
[nltk_data]   Package punkt is already up-to-date!



===== MÉTRICAS MÉDIAS =====
BLEU: 0.1099
ROUGE-1: 0.3972
ROUGE-L: 0.3155
BERTScore aprox. (cosine sim): 0.5672

Exemplos com menor BERTScore:

#1 - BERTScore (aprox) F1: 0.3564
Pergunta: Posso usar o telemóvel enquanto conduzo, se for em alta-voz?
Esperado: Sim, pode utilizar o telemóvel com sistema mãos-livres, como o modo alta-voz ou bluetooth integrado. O que é proibido é segurar ou manipular o telemóvel durante a condução.
Gerado: Sim, desde que não obstrua a visão ou distraia os outros condutores.

#2 - BERTScore (aprox) F1: 0.3726
Pergunta: Posso circular com música alta no carro?
Esperado: Não é proibido ouvir música, mas o volume excessivo que incomode os outros ou impeça a perceção de sinais sonoros pode ser motivo de coima por perturbação ou condução negligente.
Gerado: Sim, desde que não prejudique a perceção do trânsito e respeite os limites de velocidade.

#3 - BERTScore (aprox) F1: 0.6427
Pergunta: Posso conduzir de chinelos?
Esperado: Não é proibido, mas é desaconselhad

# Avaliação Manual das Respostas Geradas

Segue abaixo a avaliação manual em formato de grelha. Cada resposta foi pontuada de 0 a 5 em cinco critérios: Correcção, Clareza, Coerência, Relevância, Fluência. Os dados são referentes ao ficheiro avaliacao_dataset.json que se encontra neste diretorio.

### Tabela de Avaliações

| # | Pergunta                                                  | Correcção | Clareza | Coerência | Relevância | Fluência | Comentário                                                                                                            |
|---|-----------------------------------------------------------|-----------|---------|-----------|------------|----------|-----------------------------------------------------------------------------------------------------------------------|
| 1 | Posso conduzir de chinelos?                               | 4         | 4       | 3         | 4          | 4        | Menos factual que o gold, e menciona "capacete" que não tem nos exemplos.                 |
| 2 | Crianças podem ir no banco da frente?                     | 5         | 5       | 5         | 5          | 5        | Perfeita. Dados corretos, frase clara, bem estruturada.exame.                                              |
| 3 | Posso conduzir sem carta de condução?                     | 5         | 5       | 5         | 5          | 5        | Absolutamente correta. Mais completa até do que o gold.                                                               |
| 4 | Posso usar o telemóvel enquanto conduzo, se for em alta-voz? | 3         | 4       | 3         | 3          | 4        | Ambígua. Parece falar de "obstrução visual" mas não menciona o essencial: uso de sistema mãos-livres.                 |
| 5 | Posso circular com música alta no carro?                  | 2         | 4       | 3         | 2          | 4        | Muito permissiva. Ignora a nuance legal de ruído excessivo ser punível. Está bem escrita, mas correta.        |

### Médias

| Critério   | Média |
|------------|-------|
| Correcção  | 3.8   |
| Clareza    | 4.4   |
| Coerência  | 3.8   |
| Relevância | 3.8   |
| Fluência   | 4.4   |

### Conclusão final

O modelo escreve bem, mas ainda omite detalhes legais importantes. Quando acerta, acerta em cheio (respostas 2 e 3). 
Mas quando falha, é por ser demasiado permissivo ou impreciso, o que num domínio regulado como o Código da Estrada é algo a melhorar.
